# Feature Selection : Wrapper Method

Feature selection is the process of choosing the most relevant features (columns) for your machine learning model. It helps to:

- Improve model performance

- Reduce overfitting

- Speed up training time

- Make models more interpretable

**Feature Selection Methods:**

| Type         | Examples                                    | Description                                    |
| ------------ | ------------------------------------------- | ---------------------------------------------- |
| **Filter**   | Correlation, Chi-square, Mutual Information | Fast, statistical methods independent of model |
| **Wrapper**  | RFE, Forward/Backward Selection             | Use model performance to evaluate subsets      |
| **Embedded** | Lasso, Tree-based Feature Importance        | Selection happens during model training        |


- **Wrapper:**
    - **Recursive Feature Elimination (RFE)**: Fits the model, ranks features by importance, and eliminates the least important features iteratively.
    - **Forward Selection**: Starts with no features and adds one feature at a time that improves model performance until no further improvement.
    - **Backward Elimination**: Starts with all features and removes the least significant feature iteratively until the performance starts to degrade.

**Q.** With feature selction aren't we selecting the models as well , cause we are validating their performance on fetaures simultaneously ?

Short Answer:

> Feature selection optimizes the input space for a given model.
>
> Model selection optimizes the hypothesis space itself.

**See Run Results :** [https://dagshub.com/Rahul-404/heart-stroke-prediction.mlflow](https://dagshub.com/Rahul-404/heart-stroke-prediction.mlflow/#/)

In [ ]:
# KAGGLE_CONFIG: execute = true
# KAGGLE_CONFIG: slug = "healthcare-stroke-prediction"
# KAGGLE_CONFIG: language = "python"
# KAGGLE_CONFIG: kernel_type = "notebook"
# KAGGLE_CONFIG: is_private = true
# KAGGLE_CONFIG: enable_gpu = false
# KAGGLE_CONFIG: enable_tpu = false
# KAGGLE_CONFIG: enable_internet = true
# KAGGLE_CONFIG: machine_shape = ""
# KAGGLE_CONFIG: dataset_sources = ["rahulshelke98/healthcare-dataset-stroke-data-csv"]
# KAGGLE_CONFIG: competition_sources = []
# KAGGLE_CONFIG: kernel_sources = ["rahulshelke98/01-02-data-quality-correction", "rahulshelke98/02-03-handling-missing-values", "rahulshelke98/02-04-handling-outliers"]
# KAGGLE_CONFIG: model_sources = []
# KAGGLE_CONFIG: keywords = ["healthcare", "classification", "data-cleaning"]

In [ ]:
from datetime import datetime, timezone
print('executed at:',datetime.now(timezone.utc).isoformat())

## Required Installtion

In [ ]:
import os
import sys

# Check if running in Kaggle Cloud Environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    print("--- Kaggle Environment Detected: Installing Cloud Dependencies ---")
    # 1. Install general cloud dependencies
    os.system(f"{sys.executable} -m pip install -q mlflow dagshub")
    
    # 2. Clone the specific branch of the repository
    print("Cloning repository...")
    # os.system("git clone -b feat/notebooks https://github.com/Rahul-Shelke-1/heart-stroke-risk-stratification.git")
    os.system(f"{sys.executable} -m pip install -q git+https://github.com/Rahul-Shelke-1/heart-stroke-risk-stratification.git@feat/notebooks")
    
else:
    print("--- Local/Non-Kaggle Environment Detected: Skipping Installation ---")
    # Local packages should ideally be pre-managed via your local uv environment

## Smart Environment Detection & Secrets Setup

In [ ]:
# PRODUCTION MLOPS NOTEBOOK
# Note: This notebook runs automatically in an headless CI/CD pipeline.
# If you fork this notebook to run it yourself, make sure to set up your 
# own DagsHub repository variables or track experiments locally.

import os
import mlflow
from dotenv import load_dotenv

# 1. Safely retrieve credentials from Kaggle's internal secure vault
try:
    # 1. Detect if running inside Kaggle's container
    IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

    if IS_KAGGLE:
        DAGSHUB_USERNAME = os.environ.get("DAGSHUB_USERNAME")
        DAGSHUB_TOKEN = os.environ.get("DAGSHUB_TOKEN")
    else:
        load_dotenv()
        DAGSHUB_USERNAME = os.environ.get("DAGSHUB_USERNAME", "Rahul-404")
        DAGSHUB_TOKEN = os.environ.get("DAGSHUB_TOKEN")
    
    # Replace this with your actual DagsHub repo name
    REPO_NAME = "heart-stroke-prediction"
    
    # 2. Inject environment variables that the MLflow client natively looks for
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    
    # 3. Set the remote tracking URI to point to DagsHub
    tracking_uri = f"https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow"
    mlflow.set_tracking_uri(tracking_uri)
    
    print("Successfully connected to DagsHub MLflow tracking server!")
except Exception as e:
    # print(f"Local or non-Kaggle execution environment detected: {e}")
    print(f"Error establishing MLflow/DagsHub context: {e}")

## Unified Path Setup Strategy

In [ ]:
import os
from pathlib import Path

# 1. Reuse your environment checker flag
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    # --- KAGGLE CLOUD PATH CONTEXT ---
    # Fetch environment variables injected by your Kaggle configuration
    KAGGLE_USER = os.environ.get("KAGGLE_USER")
    DATASET_SLUG = os.environ.get("KAGGLE_DATASET_NAME")
    NOTEBOOK_1 = "01-02-data-quality-correction"
    NOTEBOOK_2 = "02-03-handling-missing-values"
    NOTEBOOK_3 = "02-04-handling-outliers"
    NOTEBOOK_4 = "03-feature-engineering"

    # Kaggle mounts datasets directly under /kaggle/input/dataset-slug
    INPUT_DATA_DIR = Path(f"/kaggle/input/datasets/{KAGGLE_USER}/{DATASET_SLUG}")
    INPUT_ARTIFACT_1_DIR = Path(f"/kaggle/input/notebooks/{KAGGLE_USER}/{NOTEBOOK_1}")
    INPUT_ARTIFACT_2_DIR = Path(f"/kaggle/input/notebooks/{KAGGLE_USER}/{NOTEBOOK_2}")
    INPUT_ARTIFACT_3_DIR = Path(f"/kaggle/input/notebooks/{KAGGLE_USER}/{NOTEBOOK_3}")
    INPUT_ARTIFACT_4_DIR = Path(f"/kaggle/input/notebooks/{KAGGLE_USER}/{NOTEBOOK_4}")

    if not INPUT_DATA_DIR.exists():
        raise FileNotFoundError(f"Kaggle Input Directory does not exist: {INPUT_DATA_DIR}")

    if not INPUT_ARTIFACT_1_DIR.exists():
            raise FileNotFoundError(f"Kaggle Artifact Directory does not exist: {INPUT_ARTIFACT_1_DIR}")
        
    if not INPUT_ARTIFACT_2_DIR.exists():
        raise FileNotFoundError(f"Kaggle Artifact Directory does not exist: {INPUT_ARTIFACT_2_DIR}")
    
    if not INPUT_ARTIFACT_3_DIR.exists():
        raise FileNotFoundError(f"Kaggle Artifact Directory does not exist: {INPUT_ARTIFACT_3_DIR}")
    
    if not INPUT_ARTIFACT_4_DIR.exists():
        raise FileNotFoundError(f"Kaggle Artifact Directory does not exist: {INPUT_ARTIFACT_4_DIR}")
    
    INPUT_DIR = {
        'data': INPUT_DATA_DIR,
        'data_correct': INPUT_ARTIFACT_1_DIR,
        'handle_missing': INPUT_ARTIFACT_2_DIR,
        'handle_missing_val': INPUT_ARTIFACT_2_DIR,
        'handle_outlier': INPUT_ARTIFACT_3_DIR,
        'engineering_pipeline': INPUT_ARTIFACT_4_DIR
    }
    
    # Kaggle strictly allows file writing ONLY inside /kaggle/working
    OUTPUT_DIR = Path("/kaggle/working")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

else:
    # --- LOCAL ENVIRONMENT PATH CONTEXT ---
    current_directory = Path.cwd()

    # Keeps your local repo clean by saving local run artifacts into an artifacts folder
    OUTPUT_DIR = current_directory / "notebooks" / "artifact"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Points to your local repository structure (e.g., repository_root/notebooks/data/)
    INPUT_DATA_DIR = current_directory / "notebooks" / "data"
    INPUT_ARTIFACT_DIR = current_directory / "notebooks" / "artifact"

    if not INPUT_DATA_DIR.exists():
        raise FileNotFoundError(f"Local Input Directory does not exist: {INPUT_DATA_DIR}")

    if not INPUT_ARTIFACT_DIR.exists():
        raise FileNotFoundError(f"Local Artifact Directory does not exist: {INPUT_ARTIFACT_DIR}")

    INPUT_DIR = {
        'data': INPUT_DATA_DIR,
        'data_correct': INPUT_ARTIFACT_DIR,
        'handle_missing': INPUT_ARTIFACT_DIR,
        'handle_missing_val': INPUT_ARTIFACT_DIR,
        'handle_outlier': INPUT_ARTIFACT_DIR,
        'engineering_pipeline': INPUT_ARTIFACT_DIR
    }

print("--- PATH SUMMARY ---")
print(f"Execution Target:    {'Kaggle Cloud' if IS_KAGGLE else 'Local Machine'}")
print(f"Reading Input From:  {INPUT_DIR}")
print(f"Writing Outputs To: {OUTPUT_DIR}")

In [ ]:
DATA_FILE_NAME = {
    'raw': Path(INPUT_DIR['data'], 'healthcare-dataset-stroke-data.csv'),
    'clean': Path(INPUT_DIR['data_correct'], 'heart_stroke_data.csv'),
}

ARTIFACT_FILE_NAME = {
    'data_correct': Path(INPUT_DIR['data_correct'], 'data_correction_pipeline.pkl'),
    'handle_missing': Path(INPUT_DIR['handle_missing'], 'missing_imputer_pipeline.pkl'),
    'handle_missing_val': Path(INPUT_DIR['handle_missing_val'], 'missing_imputer_validation_pipeline.pkl'),
    'handle_outlier': Path(INPUT_DIR['handle_outlier'], 'outlier_handler_pipeline.pkl'),
    'engineering_pipeline': Path(INPUT_DIR['engineering_pipeline'], 'feature_engineering_pipeline.pkl'),
}

### Set Experiment Name

In [ ]:
experiment_name = "04_02_Feature_Selection_FFS"
experiment_tags = {
    "experiment_type": "Classification",
    "task": "feature_selection",
    "project": "heart_stroke_risk_stratification",
    "team": "Data_Science_Core"
}

try:
    experiment_id = mlflow.create_experiment(name=experiment_name, tags=experiment_tags)
except Exception:
    # If it already exists, fetch its ID
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# 3. Set it as the active experiment
mlflow.set_experiment(experiment_id=experiment_id)

### Set Run Name

In [ ]:
# Starts a global active run
mlflow.start_run(run_name="Forward_Feature_Selection_Logistic_Regression")

In [ ]:
mlflow.autolog()

## **1. Imports**

In [1]:
from heart_stroke_prediction.analyze.feature_selection.wrapper_methods import Wrapper_Methods
import pandas as pd
import cloudpickle
import os

from sklearn import set_config
# Set global configuration to output Pandas DataFrames
set_config(transform_output="pandas")

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'cudf'

In [ ]:
DATA_FILE_NAME = {
    'raw': Path(INPUT_DIR['data'], 'healthcare-dataset-stroke-data.csv'),
    'clean': Path(INPUT_DIR['data_correct'], 'heart_stroke_data.csv'),
}

ARTIFACT_FILE_NAME = {
    'data_correct': Path(INPUT_DIR['data_correct'], 'data_correction_pipeline.pkl'),
    'handle_missing': Path(INPUT_DIR['handle_missing'], 'missing_imputer_pipeline.pkl'),
    'handle_missing_val': Path(INPUT_DIR['handle_missing_val'], 'missing_imputer_validation_pipeline.pkl'),
    'handle_outlier': Path(INPUT_DIR['handle_outlier'], 'outlier_handler_pipeline.pkl'),
    'engineering_pipeline': Path(INPUT_DIR['engineering_pipeline'], 'feature_engineering_pipeline.pkl'),
}

## **2. Static Pipeline**

In [ ]:
with open(ARTIFACT_FILE_NAME["data_correct"], "rb") as f:
    static_pipeline = cloudpickle.load(f)

In [34]:
static_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('correct_column_names', ...), ('correct_column_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,mapping,{'Residence_type': 'residence_type'}
,column_mappings,"{'ever_married': {'No': 'no', 'Yes': 'yes'}, 'gender': {'Female': 'female', 'Male': 'male', 'Other': 'other'}, 'residence_type': {'Rural': 'rural', 'Urban': 'urban'}, 'smoking_status': {'Unknown': 'unknown', 'formerly smoked': 'formerly_smoked', 'never smoked': 'never_smoked'}, ...}"
,missing_tokens,['unknown']
,threshold,1
,action,'drop'
,fill_value,'other'


## **3. Read Data**

In [ ]:
df = pd.read_csv(DATA_FILE_NAME['raw'])
df = static_pipeline.fit_transform(df)

In [36]:
df.shape

(5109, 12)

In [37]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


In [38]:
df.isna().sum()

id                      0
gender                  0
age                     0
hypertension            0
heart_disease           0
ever_married            0
work_type               0
residence_type          0
avg_glucose_level       0
bmi                   201
smoking_status       1544
stroke                  0
dtype: int64

In [39]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')

## **4. Column Seperation**

In [40]:
TARGET_COL = ['stroke']
ID_COL = ['id']
NUM_FEATURES = ['age', 'bmi', 'avg_glucose_level']
CAT_FEATURES = [ 'gender', 'ever_married', 'work_type', 'residence_type', 'smoking_status', 'hypertension', 'heart_disease']

## **5. Dynamic Pipeline**

In [ ]:
with open(ARTIFACT_FILE_NAME["engineering_pipeline"], "rb") as f:
    dynamic_pipeline = cloudpickle.load(f)

In [42]:
dynamic_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute_missing', ...), ('feature_extraction', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [43]:
# feature_engineering_pipeline = joblib.load(os.path.join(ARTIFACT_PATH, "feature_engineering_pipeline_v1.pkl"))
df_composed = dynamic_pipeline.fit_transform(df)
df_composed[TARGET_COL] = df[TARGET_COL]

In [44]:
df_composed.head()

,gender_male,ever_married_yes,residence_type_urban,work_type_children,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,smoking_status_formerly_smoked,smoking_status_missing,...,bmi_age_ratio,avg_glucose_bmi_ratio,age_sqrt,bmi_sqrt,avg_glucose_level_sqrt,hypertension,heart_disease,age,missingindicator_bmi,stroke
0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.546269,8370.054,8.185353,6.049793,15.122500,0,1,67.0,0.0,1
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.460656,5682.101,7.810250,5.300943,14.220056,0,0,61.0,1.0,1
2,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.406250,3442.400,8.944272,5.700877,10.291744,0,1,80.0,0.0,1
3,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.702041,5890.312,7.000000,5.865151,13.085488,0,0,49.0,0.0,1
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.303797,4178.880,8.888194,4.898979,13.195454,1,0,79.0,0.0,1


In [45]:
df_composed.columns

Index(['gender_male', 'ever_married_yes', 'residence_type_urban',
       'work_type_children', 'work_type_govt_job', 'work_type_never_worked',
       'work_type_private', 'work_type_self_employed',
       'smoking_status_formerly_smoked', 'smoking_status_missing',
       'smoking_status_never_smoked', 'smoking_status_smokes',
       'missingindicator_smoking_status_True', 'bmi', 'avg_glucose_level',
       'bmi_health_category', 'glucose_health_category',
       'age_avg_glucose_interaction', 'age_bmi_interaction', 'bmi_age_ratio',
       'avg_glucose_bmi_ratio', 'age_sqrt', 'bmi_sqrt',
       'avg_glucose_level_sqrt', 'hypertension', 'heart_disease', 'age',
       'missingindicator_bmi', 'stroke'],
      dtype='object')

In [46]:
df_composed.shape

(5109, 29)

## **6. Feature Set**

In [47]:
NUM_FEATURES = ['bmi', 'avg_glucose_level','age_avg_glucose_interaction', 'age_bmi_interaction', 'bmi_age_ratio',
                'avg_glucose_bmi_ratio', 'age_sqrt', 'bmi_sqrt', 'avg_glucose_level_sqrt','age', 'missingindicator_bmi']
CAT_FEATURES = ['gender_male', 'ever_married_yes', 'residence_type_urban',
                'work_type_children', 'work_type_govt_job', 'work_type_never_worked',
                'work_type_private', 'work_type_self_employed',
                'smoking_status_formerly_smoked', 'smoking_status_missing',
                'smoking_status_never_smoked', 'smoking_status_smokes',
                'missingindicator_smoking_status_True', 'bmi_health_category', 
                'glucose_health_category','hypertension', 'heart_disease',]

In [48]:
features = NUM_FEATURES + CAT_FEATURES + TARGET_COL

# Feature Selection Method : Wrapper Method

In [49]:
import time
from typing import Dict, List, Optional, Tuple, Any
# import cudf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import make_scorer
from sklearn.metrics import roc_auc_score, average_precision_score
from src.analyze.registry import MODEL_REGISTRY
# from src.analyze.feature_selection.wrapper_methods import Wrapper_Methods
import cloudpickle

# Filter out the specific Scikit-Learn Parallel/delayed UserWarning
import warnings
warnings.filterwarnings("ignore")

In [2]:
class Wrapper_Methods:
    """
    This wrapper method is customized to handle classification problem with 5% prevalanced in data.
    """

    def __init__(
            self,
            df: pd.DataFrame,
            features: List[str],
            target: List[str],
            preprocessor: Pipeline,
            scaler_type: str = "standard",
            model_type: str = "classification",
            shuffle: bool = True,
            n_splits:int = 5,
            n_repeats: int = 10,
            seed: int = 42,
            bins: int = 10,
            use_gpu: bool = False
        ):
        # data param
        self.df = df                                # dataframe
        self.features = features                    # feature from dataframe
        self.target = target                        # target name from dataframe
        # model param
        self.preprocessor = preprocessor            # preprocessor pipeline
        self.scaler_type = scaler_type              # scalar to scale data
        self.model_type = model_type                # type of problem
        # experiment param
        self.shuffle = shuffle                      # shiffle data before making split
        self.n_splits = n_splits                    # The standard 5 folds
        self.n_repeats = n_repeats                  # Your N unique experiments
        self.seed = seed                            # One seed controls all N repetitions
        self.bins = bins                            # binning for calibration
        # tags/abbrivations of all regression models
        self.model_clf_tags = ["LR", "RDG", "NB", "SVM", "KNN", "DT", "RF", "SGD", "GB", "AB", "ETC", "XGB", "XGBRF", "LGB", "CB"]
        self.model_report = dict()
        self.use_gpu = use_gpu

        self._validate_parameters()

        self.X, self.y = self._load_data()

    def _validate_parameters(self):
        """Validate initialization parameters."""

        valid_scalers = ['minmax', 'standard']
        valid_model_type = ['classification', 'regression']

        # data param validation
        if not isinstance(self.df, pd.DataFrame):
            raise TypeError(f"df must be a pd.DataFrame, got {type(self.df)}")

        if not isinstance(self.features, list):
            raise TypeError(f"features must be a list of string, got {type(self.features)}")

        for col in self.features:
            if not isinstance(col, str):
                warnings.warn(f"Column '{col}' must be a str, got {type(col)}")
            elif col not in self.df.columns.to_list():
                warnings.warn(f"Column '{col}' is not in dataframe")

        if not isinstance(self.target, list):
            raise TypeError(f"target must be a list, got {type(self.target)}")

        for col in self.target:
            if not isinstance(col, str):
                warnings.warn(f"Column '{col}' must be a str, got {type(col)}")
            elif col not in self.df.columns.to_list():
                warnings.warn(f"Column '{col}' is not in dataframe")

        # model param validation
        if not isinstance(self.preprocessor, Pipeline):
            raise TypeError(f"preprocessor must be sklearn.pipeline, got '{type(self.preprocessor)}'")

        if not isinstance(self.scaler_type, str):
            raise TypeError(f"scaler_type must be a str, got {type(self.scaler_type)}")

        if self.scaler_type not in valid_scalers:
            warnings.warn(
                f"scaler_type '{self.scaler_type}' is not recognized. "
                f" Valid options are: {valid_scalers}"
            )

        if not isinstance(self.model_type, str):
            raise TypeError(f"model_type must be a str, got {type(self.model_type)}")

        if self.model_type not in valid_model_type:
            warnings.warn(
                f"model_type '{self.model_type}' is not recognized. "
                f" Valid options are: {valid_model_type}"
            )

        # experiment param validation
        if not isinstance(self.shuffle, bool):
            raise TypeError(f"shuffle must be bool, got {type(self.shuffle)}")

        if not isinstance(self.n_splits, int):
            raise TypeError(f"n_splits must be int, got {type(self.n_splits)}")

        if not isinstance(self.n_repeats, int):
            raise TypeError(f"n_repeats must be int, got {type(self.n_repeats)}")

        if not isinstance(self.seed, int):
            raise TypeError(f"seed must be int, got {type(self.seed)}")

        if not isinstance(self.bins, int):
            raise TypeError(f"bins must be int, got {type(self.bins)}")

    def _generate_random_seeds(
        self,
        base_seed: int,
        n_seeds: int,
        low: int = 0,
        high: int = 2**32 - 1,
        ) -> List[int]:
        """
        Generate reproducible random seeds from a base seed.

        Parameters
        ----------
        base_seed : int
            Master seed for reproducibility
        n_seeds : int
            Number of random seeds to generate
        low : int
            Minimum seed value (inclusive)
        high : int
            Maximum seed value (exclusive)

        Returns
        -------
        List[int]
            List of deterministic random seeds
        """
        rng = np.random.default_rng(base_seed)
        return rng.integers(low=low, high=high, size=n_seeds).tolist()

    def _load_data(self) -> Tuple[pd.DataFrame, pd.Series]:
        try:
            if self.use_gpu:
              import cudf
              X = cudf.from_pandas(self.df[self.features])
              y = cudf.from_pandas(self.df[self.target[0]])
              print("Moving data to GPU for accelerated feature selection...")
            else:
              X = self.df[self.features]
              y = self.df[self.target[0]]

            return X, y
        except Exception as e:
            raise e

    def _load_preprocessor(self, random_state: int = None) -> ImbPipeline:
        try:
            # assemble model
            steps = []

            # 2. scaler
            if self.scaler_type == 'standard':
                full_pipeline = ImbPipeline(
                    steps=self.preprocessor.steps+[('scaler', StandardScaler())]
                    )
            elif self.scaler_type == 'minmax':
                full_pipeline = ImbPipeline(
                    steps=self.preprocessor.steps+[('scaler', MinMaxScaler())]
                    )
            else:
                full_pipeline = ImbPipeline(steps=self.preprocessor.steps)

            return full_pipeline
        except Exception as e:
            raise RuntimeError(f"Error in _load_preprocessor : {e}")

    def _get_model(
            self,
            tag: str,
            params: dict | None = None
        ) -> [str, ImbPipeline, Any]:
        try:
            # 1. Validate model_type
            if self.model_type not in MODEL_REGISTRY:
                raise KeyError(f"Unknown model_type: '{self.model_type}'")

            registry = MODEL_REGISTRY[self.model_type]

            # 2. Validate tag existence
            if tag not in registry:
                raise KeyError(f"Unknown model_tag: '{tag}'")
            # 3. get model name and model
            model_name, model_cls = registry[tag]
            # 4. update model params
            model = model_cls(**(params or {}))

            # assemble model
            steps = []

            # 1. scaler
            if self.scaler_type == 'standard':
                steps.append(('scaler', StandardScaler()))
            else:
                steps.append(('scaler', MinMaxScaler()))

            full_pipeline = ImbPipeline(steps=self.preprocessor.steps+steps)

            full_pipeline.set_output(transform="pandas")

            return model_name, full_pipeline, model
        except Exception as e:
            raise RuntimeError(f"Error in _get_model: {e}")

    def recall_at_percent_func(self, y_true, y_probs, percent):
        # Ensure we use probability of the positive class (column index 1)
        if len(y_probs.shape) == 2:
            y_probs = y_probs[:, 1]

        n = len(y_true)
        k = int(np.ceil(n * percent))
        # Sort indices by descending probability
        indices = np.argsort(y_probs)[::-1]
        top_k = indices[:k]

        # Handle pandas series vs numpy arrays
        y_true_array = y_true.values if hasattr(y_true, "values") else y_true
        return y_true_array[top_k].sum() / y_true_array.sum()

    def precision_at_percent_func(self, y_true, y_probs, percent):
        if len(y_probs.shape) == 2:
            y_probs = y_probs[:, 1]

        n = len(y_true)
        k = int(np.ceil(n * percent))
        indices = np.argsort(y_probs)[::-1]
        top_k = indices[:k]

        y_true_array = y_true.values if hasattr(y_true, "values") else y_true
        return y_true_array[top_k].sum() / k

    def nns_at_percent_func(self, y_true, y_probs, percent):
        # NNS is 1 / Precision
        precision = self.precision_at_percent_func(y_true, y_probs, percent)
        if precision == 0:
            return np.inf  # Lower is better for NNS
        return 1 / precision

    def _evaluate_selected_features(self, X_train, y_train, X_test, y_test, model):
        """
        Docstring for _evaluate_selected_features
        """
        try:
            metrics = {
                "roc_auc": [],
                "pr_auc": [],
                "recall_10": [],
                "recall_15": [],
                "recall_20": [],
                "precision_10": [],
                "nns_10": []
            }

            # fit the model
            model.fit(X_train, y_train)
            y_scores = model.predict_proba(X_test)[:, 1]

            # Core metrics
            metrics["roc_auc"].append(roc_auc_score(y_test, y_scores))
            metrics["pr_auc"].append(average_precision_score(y_test, y_scores))

            # Ranking metrics
            metrics["recall_10"].append(self.recall_at_percent_func(y_test.values, y_scores, 0.10))
            metrics["recall_15"].append(self.recall_at_percent_func(y_test.values, y_scores, 0.15))
            metrics["recall_20"].append(self.recall_at_percent_func(y_test.values, y_scores, 0.20))

            metrics["precision_10"].append(self.precision_at_percent_func(y_test.values, y_scores, 0.10))
            metrics["nns_10"].append(self.nns_at_percent_func(y_test.values, y_scores, 0.10))

            return metrics
        except Exception as e:
            raise RuntimeError(f"Error in _evaluate_model: {e}")

    def _print_scores(self, metrics: Dict, model_name:str, selected_features: List[str]):
        try:
            # print model name
            print('='*15, model_name,'='*15)

            print(f"\nSelected Fetaures: {selected_features}\n")

            for key, values in metrics['all_scores'][0].items():
                print(f"{key}: {np.mean(values):.4f} ({np.std(values):.4f})")

            print()
        except Exception as e:
            raise e

    def box_plot(self, names: Optional[List[str]] = None, results: Optional[List[List[float]]]= None) -> None:
        """
        Plots a box plot to compare model performance across different feature combinations.

        Args:
            names (List[str]): List of model or feature combination names (used as x-axis labels).
            results (List[List[float]]): A list of score lists, where each sublist contains
                                        cross-validation scores for a model or feature set.

        Returns:
            None

        Raises:
            ValueError: If the input lengths of `names` and `results` do not match.
            Exception: For any unexpected errors during plotting.
        """
        try:
            if names is None and results is None:
                names = self.model_report.keys()
                results = self.model_report.values()

            if len(names) != len(results):
                raise ValueError("Length of 'names' must match the number of result sets.")

            plt.boxplot(results, labels=names, showmeans=True, vert=False)
            plt.ylabel("Models")
            plt.xlabel(f"Score: {self.primary_scoring}")
            plt.title("Performance of Models")
            plt.grid()
            plt.show()

        except Exception as e:
            print(f"An error occurred while plotting the box plot: {e}")
            raise

    def get_report_to_df(self, scores: List[List[Dict]]):
        try:
            df = dict()

            # model name
            df['model_name'] = scores["model_name"]

            # scores
            for key, value in scores['all_scores'][0].items():
                df[key] = np.mean(value)

            # selected features
            df['selected_features'] = [feature_array.tolist() for feature_array in scores["selected_features"]]

            # return sorted report
            return pd.DataFrame(df).sort_values(by="recall_10", ascending=False).reset_index(drop=True)
        except Exception as e:
            raise e
    
    def BalancedBaggingClassifierCustom(self):
        """GPU acceleration
        
        here we only pass 
        logistic regression
        support vector machine
        k nearest neighbor
        naive bayes
        """
        pass

    def BalancedBaggingRegressorCustom(self):
        """GPU acceleration"""
        pass

    def directional_feature_selection(
        self,
        model_configs: Optional[Dict[str, Optional[dict]]] = None,
        print_metric: bool = True,
        direction: str = "forward",
    ) -> dict:
        """
        Objective: this works on subset of features and there influence over evaluation metric, that will get us to max score
        by forwardly or backwardly eliminating the features from superset

        Remainder: this technique works with parametric and non-parametric models, does not rely on feature importance or
        models coefficients to determin the feature selection

        model_configs (Dict[str, Optional[dict, Any]]): it contains model tag, and model parameters (optional)

        print_status (bool): print model training status

        direction (str): feature selection direction , default is "forward"
                            "forward"  : subset -> superset
                            "backward" : superset -> subset

        Returns:

        model_report (dict): models performance on each subset of features
        """
        try:
            model_report = {
                "model_name": [],
                "selected_features": [],
                "all_scores": [
                    {
                        "roc_auc": [],
                        "pr_auc": [],
                        "recall_10": [],
                        "recall_15": [],
                        "recall_20": [],
                        "precision_10": [],
                        "nns_10": [],
                    }
                ],
            }

            # --------- Select Features -------------

            for model_tag, params in model_configs.items():

                # ------------- sequenctial feature selection ------------------

                model_name, full_pipeline, model = self._get_model(model_tag, params, None)

                if self.model_type == "classification":
                    # for cross validation classification
                    cv = StratifiedKFold(n_splits=self.n_splits, shuffle=self.shuffle, random_state=self.seed)
                else:
                    # for cross validation regression
                    cv = StratifiedKFold(n_splits=self.n_splits, shuffle=self.shuffle, random_state=self.seed)

                # balanced bagging classifier with base estimator
                bbc = BalancedBaggingClassifier(
                            estimator=model, # Assuming the last step is the base model
                            sampling_strategy='auto',
                            random_state=self.seed,
                            n_jobs=-1
                        )

                # This ensures SFS selects features based on CALIBRATED Precision/F2
                search_estimator = CalibratedClassifierCV(
                    estimator=bbc,
                    method='sigmoid',
                    cv=cv, # Use the same CV for calibration logic
                )

                # Create the scorers
                recall_pct_scorer = make_scorer(
                    self.recall_at_percent_func,
                    percent=0.10,
                    response_method="predict_proba" # or needs_proba=True in older versions
                )

                # features selection object
                selection_obj = SequentialFeatureSelector(
                    estimator=search_estimator,   # try on this model
                    n_features_to_select='auto',  # select automatically
                    tol = None,                   # score threshould
                    direction=direction,          # direction of feature selection
                    scoring=recall_pct_scorer,    # evaluation metric
                    cv=cv,                        # cross validation method
                    n_jobs=1,                    # use all cores
                )

                # combining pipeline and model
                full_model = ImbPipeline(steps=full_pipeline.steps + [("sfs", selection_obj)])

                # print(f"full model: {full_model.named_steps.keys()}")

                start = time.time()
                # 2. Fit the selector
                full_model.fit(self.X, self.y)
                end = time.time()
                print("Sequential Feature Selection Execution time:", end - start, "seconds")

                # 3. FIX: Index safely using NumPy
                selected_indices = full_model['sfs'].get_support(indices=True)

                # print(f"features indices: {selected_indices}")

                # print(f"{full_model.named_steps}")

                feature_names = full_model.named_steps["encoding"].get_feature_names_out()
                selected_features = feature_names[selected_indices]

                # print(f"features selected: {selected_features}")


                # --------- validate the features robustness ------------

                seed_values = self._generate_random_seeds(
                                    base_seed=self.seed,
                                    n_seeds=self.n_repeats
                                )

                eval_scores_per_exp = []
                features_list_per_exp = []

                for seed in seed_values:

                    # splitting the data into train test split
                    X_train_sandbox, X_holdout, y_train_sandbox, y_holdout = train_test_split(
                        self.X, self.y, test_size=0.3, random_state=seed,
                        shuffle=self.shuffle, stratify=self.y
                    )

                    # 1. Transform data through the preprocessing parts of the pipeline
                    X_train_processed = full_pipeline.fit_transform(X_train_sandbox)
                    X_val_processed = full_pipeline.transform(X_holdout)

                    # ------------- evaluating selected feature ------------------
                    scores = self._evaluate_selected_features(
                                        X_train_processed.iloc[:, selected_indices],
                                        y_train_sandbox,
                                        X_val_processed.iloc[:, selected_indices],
                                        y_holdout,
                                        bbc
                                    )

                    # eval_scores_per_exp[0][].append(scores["roc_auc"])
                    model_report["all_scores"][0]["roc_auc"].append(scores["roc_auc"])
                    model_report["all_scores"][0]["pr_auc"].append(scores["pr_auc"])
                    model_report["all_scores"][0]["recall_10"].append(scores["recall_10"])
                    model_report["all_scores"][0]["recall_15"].append(scores["recall_15"])
                    model_report["all_scores"][0]["recall_20"].append(scores["recall_20"])
                    model_report["all_scores"][0]["precision_10"].append(scores["precision_10"])
                    model_report["all_scores"][0]["nns_10"].append(scores["nns_10"])

                # collecting models name
                model_report["model_name"].append(model_name)

                # feature lists
                model_report["selected_features"].append(selected_features)

                # print scores with only metric from passed emtrics
                if print_metric:
                    self._print_scores(model_report, model_name, selected_features)

            return model_report
        except Exception as e:
            raise RuntimeError(f"Error in directional_feature_selection : {e}")



NameError: name 'pd' is not defined

In [51]:
# from src.analyze.feature_selection.wrapper_methods import Wrapper_Methods
import cloudpickle

In [52]:
# load data
df = static_pipeline.fit_transform(pd.read_csv(os.path.join(os.getcwd(), "data", DATA_FILE_NAME['raw'])))

In [53]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='object')

In [ ]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,male,67.0,0,1,yes,private,urban,228.69,36.6,formerly_smoked,1
1,51676,female,61.0,0,0,yes,self_employed,rural,202.21,NaN,never_smoked,1
2,31112,male,80.0,0,1,yes,private,rural,105.92,32.5,never_smoked,1
3,60182,female,49.0,0,0,yes,private,urban,171.23,34.4,smokes,1
4,1665,female,79.0,1,0,yes,self_employed,rural,174.12,24.0,never_smoked,1


In [ ]:
df.shape

(5109, 12)

In [ ]:
TARGET_COL = ['stroke']
ID_COL = ['id']
NUM_FEATURES = ['age', 'bmi', 'avg_glucose_level',]
CAT_FEATURES = [ 'gender', 'ever_married', 'work_type', 'residence_type', 'smoking_status', 'hypertension', 'heart_disease']

In [ ]:
with open(os.path.join(os.getcwd(), "artifacts", ARTIFACT_FILE_NAME["engineering_pipeline"]), "rb") as f:
    preprocessor = cloudpickle.load(f)

In [ ]:
preprocessor

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('impute_missing', ...), ('feature_extraction', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [ ]:

start = time.time()
preprocessor.fit_transform(df)
end = time.time()

print(f"Execution time: {end - start:.6f} seconds")

Execution time: 0.035394 seconds


### 2.1 Forward Feature Selection

In [ ]:
_, pos_case = train_test_split(df, test_size=0.20, stratify=df['stroke']) # positive cases
neg_case, _ = train_test_split(df, test_size=0.989, stratify=df['stroke']) # negative cases

In [ ]:
pos_case.shape, neg_case.shape

((1022, 12), (56, 12))

In [ ]:
neg_case['stroke'].value_counts()

stroke
0    53
1     3
Name: count, dtype: int64

In [ ]:
test = pd.concat([pos_case[pos_case['stroke'] == 1], neg_case[neg_case['stroke'] == 0]])

In [ ]:
params = {
    "df": test,
    "features" : ID_COL+TARGET_COL+NUM_FEATURES+CAT_FEATURES,
    "target" : TARGET_COL,
    "preprocessor": preprocessor,
    "scaler_type": "standard",
    "model_type": "classification",
    "shuffle": True,
    "n_splits": 5,
    "n_repeats": 5,
    "seed": 42,
    "bins": 10,
}

In [ ]:
model_config = {"LR": {"max_iter": 1000}}

In [ ]:
wm = Wrapper_Methods(**params)

#### Logistic Regression

In [ ]:
model_report = wm.directional_feature_selection(
                model_configs=model_config,
                print_metric=True
            )

Sequential Feature Selection Execution time: 102.08419704437256 seconds
=============== Logistic Regression ===============

Selected Fetaures: ['gender_male' 'ever_married_yes' 'residence_type_urban'
 'work_type_children' 'work_type_govt_job' 'work_type_infrequent_sklearn'
 'smoking_status_formerly_smoked' 'smoking_status_smokes' 'bmi'
 'avg_glucose_level' 'bmi_health_category' 'bmi_age_ratio'
 'avg_glucose_level_sqrt' 'heart_disease']

roc_auc: 0.7117 (0.0789)
pr_auc: 0.6944 (0.0490)
recall_10: 0.1867 (0.0267)
recall_15: 0.2400 (0.0327)
recall_20: 0.3467 (0.0499)
precision_10: 0.7000 (0.1000)
nns_10: 1.4667 (0.2667)



### Final Report

In [ ]:
report = wm.get_report_to_df(model_report)

In [ ]:
report

,model_name,roc_auc,pr_auc,recall_10,recall_15,recall_20,precision_10,nns_10,selected_features
0,Logistic Regression,0.847164,0.202591,0.461333,0.581333,0.682667,0.224675,4.518959,"[gender_male, residence_type_urban, work_type_..."
1,Decision Tree,0.800113,0.146992,0.341333,0.456000,0.573333,0.166234,6.177737,"[ever_married_yes, residence_type_urban, work_..."
2,Extra Trees Classifier,0.847995,0.201466,0.429333,0.560000,0.688000,0.209091,4.914923,"[ever_married_yes, work_type_children, work_ty..."
3,K-Neighbors Classifier,0.829777,0.188180,0.424000,0.560000,0.674667,0.206494,4.914655,"[ever_married_yes, work_type_children, work_ty..."


In [ ]:
from sklearn.pipeline import Pipeline  # pipeline

In [ ]:
ALL_FEATURES = NUM_FEATURES+CAT_FEATURES
STATISTICALLY_FILTTERED = ['aba',
 'ever_married_encode',
 'work_type_encode',
 'work_type_self_employed',
 'smoking_status_encode',
 'smoking_status_formerly_smoked',
 'smoking_status_never_smoked'] + TARGET_COL

In [ ]:
df[NUM_FEATURES+CAT_FEATURES].columns

In [ ]:
# features = df[NUM_FEATURES+CAT_FEATURES].columns
X = np.array(df[STATISTICALLY_FILTTERED].values, dtype=np.float64)
y = df[TARGET_COL].iloc[:, 0].values

In [ ]:
# linear model
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
# Byes model
from sklearn.naive_bayes import GaussianNB
# support vector machine
from sklearn.svm import SVC
# distance based model
from sklearn.neighbors import KNeighborsClassifier
# tree based
from sklearn.tree import DecisionTreeClassifier
# bagging special case
from sklearn.ensemble import RandomForestClassifier
# ensemble
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier, XGBRFClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.pipeline import Pipeline

from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import RFECV

import random

from typing import Optional, List, Tuple

In [ ]:
# write a code to execute individial model at a tmie.

In [ ]:
class Wrapper_Methods:

    def __init__(self, df:pd.DataFrame, features:List[str], target:List[str], scoring:str, cv:int, seed:int, shuffle:bool, exp_no: int):
        self.df = df
        self.features = features
        self.target = target
        self.scoring = scoring
        self.cv = cv
        self.seed = seed
        self.shuffle = shuffle
        self.exp_no = exp_no
        self.X, self.y = self.load_data()
        self.model_tags = ["LR", "RDG", "NB", "SVM", "KNN", "DT", "RF", "SGD", "GB", "AB", "ETC", "XGB", "XGRFB", "LGB", "CB"]
        self.model_report = dict()

    def load_data(self):
        try:
            X = self.df[self.features].values
            # y = self.df[self.target].iloc[:,0].values
            y = self.df[self.target].values.ravel() 
            return X, y
        except Exception as e:
            raise e

    def get_model(self, model_tag):
        try:
            if model_tag == "LR":
                return {"LR": ("Logistic Regression", LogisticRegression(solver="liblinear" ,max_iter=1000))}
            
            if model_tag == "RDG":
                return {"RDG": {"Ridge Classifier", RidgeClassifier()}}
            
            if model_tag == "SGD":
                return {"SGD": ("Stocastic Gradient", SGDClassifier())}
            
            if model_tag == "NB":        
                return {"NB": ("Naive Bayes", GaussianNB())}

            if model_tag == "SVM":        
                return {"SVM": ("Support Vector", SVC(kernel='linear'))} # this kernel works with RFE

            if model_tag == "KNN":         
                return {"KNN": ("K-Neighbors", KNeighborsClassifier(n_neighbors=2))}

            if model_tag == "DT":        
                return {"DT": ("Decision Tree", DecisionTreeClassifier())}

            if model_tag == "RF":        
                return {"RF": ("Random Forest", RandomForestClassifier())}

            if model_tag == "GB":        
                return {"GB": ("Gradient Boosting", GradientBoostingClassifier())}

            if model_tag == "AB":        
                return {"AB": ("Ada Boosting", AdaBoostClassifier())}

            if model_tag == "ETC":        
                return {"ETC": ("Extra Tres Boosting", ExtraTreesClassifier())}

            if model_tag == "XGB":        
                return {"XGB": ("XG Boost", XGBClassifier())}

            if model_tag == "XGRFB":        
                return {"XGRFB": ("XG Boost(RF)", XGBRFClassifier())}

            if model_tag == "LGB":        
                return {"LGB": ("Light Boosting",LGBMClassifier(learning_rate=0.01, n_estimators=1000))}

            if model_tag == "CB":        
                return {"CB": ("Cat Boosting", CatBoostClassifier(verbose=0))}
        except Exception as e:
            raise e

    def evaluate_model(self, seed, model, n_features:List[int]):
        try:
            # define the model evaluation procedure
            cv = StratifiedKFold(n_splits=self.cv, random_state=seed, shuffle=self.shuffle)
            # evaluate the model: n-jobs=-1 [uses all cores]
            scores = cross_val_score(model, self.X[:,n_features], self.y, scoring=self.scoring, cv=cv, n_jobs=-1)
            # return scores
            return scores
        except Exception as e:
            raise e

    def get_custome_scores(self, scores: List[float]):
        try:
            mean_score = np.mean(scores)
            std_score = np.std(scores)
            # return mean and std for cross validations scores
            return mean_score, std_score
        except Exception as e:
            raise e
    
    def box_plot(self, names: Optional[List[str]] = None, results: Optional[List[List[float]]]= None) -> None:
        """
        Plots a box plot to compare model performance across different feature combinations.

        Args:
            names (List[str]): List of model or feature combination names (used as x-axis labels).
            results (List[List[float]]): A list of score lists, where each sublist contains
                                        cross-validation scores for a model or feature set.

        Returns:
            None

        Raises:
            ValueError: If the input lengths of `names` and `results` do not match.
            Exception: For any unexpected errors during plotting.
        """
        try:
            if names is None and results is None:
                names = self.model_report.keys()
                results = self.model_report.values()

            if len(names) != len(results):
                raise ValueError("Length of 'names' must match the number of result sets.")

            plt.boxplot(results, labels=names, showmeans=True, vert=False)
            plt.ylabel("Models")
            plt.xlabel(f"Score: {self.scoring}")
            plt.title("Performance of Models")
            plt.grid()
            plt.show()

        except Exception as e:
            print(f"An error occurred while plotting the box plot: {e}")
            raise

    @staticmethod
    def get_scores(df: pd.DataFrame, columns: List[str] = ["model_name", "cv_scores"]):
        try:
            # get columns
            df = df[columns].copy()
            # get mean
            df["mean"] = df["cv_scores"].map(lambda x: np.mean(x))
            # get std
            df["std"] = df["cv_scores"].map(lambda x: np.std(x))
            # return sorted report
            return df.sort_values(by='mean', ascending=False).reset_index(drop=True)
        except Exception as e:
            raise e

    def directional_feature_selection(
        self, 
        model_tags: Optional[List[str]] = None,
        print_status:bool = False, 
        direction: str = "forward",
        custom_model: Optional[List[Tuple[str, BaseEstimator]]] = None
    ) -> dict:
        """ 
        Objective: this works on subset of features and there influence over evaluation metric, that will get us to max score
        by forwardly or backwardly eliminating the features from superset

        Remainder: this technique works with parametric and non-parametric models, does not rely on feature importance or 
        models coefficients to determin the feature selection

        Parameters:

        model_tags (List[str]): abbrivateions of model its optional 

        print_status (bool): print model training status

        direction (str): feature selection direction , default is "forward"
                            "forward"  : subset -> superset
                            "backward" : superset -> subset

        Returns:

        model_report (dict): models performance on each subset of features
        """
        try:
            # run model evaluation for all sub set of features
            model_report = dict()
            features_list = []
            model_names = []
            all_scores = []

            # Decide which models to run
            models_to_run = []

            if custom_model is not None:
                models_to_run.extend(custom_model)
            elif model_tags:
                for model_tag in model_tags:
                    model_name, model = self.get_model(model_tag)[model_tag]
                    models_to_run.append((model_name, model))
            else:
                raise ValueError("You must provide either `model_tags` or a `custom_model`.")
            
            # Generate 30 unique random seeds between 1 and 1,000,000
            unique_seeds = random.sample(range(1, 1000000), self.exp_no)
            
            for seed in unique_seeds:

                # cross validation object
                cv = StratifiedKFold(n_splits=self.cv, random_state=seed, shuffle=self.shuffle)

                # feature selection run on models  
                for model_name, model in models_to_run:

                    # features selection object
                    selection_obj = SequentialFeatureSelector(
                        estimator=model,         # try on this model
                        n_features_to_select="auto", # select automatically,
                        tol = None,                  # score threshould
                        direction=direction,         # direction of feature selection
                        scoring=self.scoring,        # evaluation metric
                        cv=cv,                       # cross validation method
                        n_jobs=-1,                   # use all cores
                    )
                    # fitting object to select features
                    selection_obj.fit(X, y)

                    # selected features
                    selected_features = selection_obj.get_feature_names_out(self.features)

                    # collecting finally selected features
                    features_list.append(selected_features)

                    # collecting models name
                    model_names.append(model_name)    

                    # evaluating for score on selected features
                    eval_scores = self.evaluate_model(model, selection_obj.get_support(indices=True))

                    # keeping the track of model run and scores
                    self.model_report[model_name] = eval_scores

                    # collecting scores of each fold
                    all_scores.append(eval_scores)

                    # print scores
                    if print_status:
                        # print model name
                        print('='*15,model_name,'='*15)

                        # mean scores
                        mean, std = self.get_custome_scores(eval_scores)

                        msg = f"{len(selected_features)} : Mean: {mean} Std: {std}\n"
                        print(msg)

            # collecting all scores
            model_report["model_name"] = model_names
            model_report["selected_features"] = features_list
            model_report["cv_scores"] = all_scores

            return model_report
        except Exception as e:
            raise e

    def backward_feature_elimination(
        self,
        model_tags: Optional[List[str]] = None,
        print_status:bool = False,
        custom_model: Optional[List[Tuple[str, BaseEstimator]]] = None
    ) -> dict:
        """ 
        Objective: this works on subset of features and there influence over evaluation metric, that will get us to max score
        by forwardly or backwardly eliminating the features from superset

        Remainder: this technique works with parametric and non-parametric models, does not rely on feature importance or 
        models coefficients to determin the feature selection

        Parameters:

        model_tags (List[str]): abbrivateions of model its optional 

        print_status (bool): print model training status

        direction (str): feature selection direction , default is "forward"
                            "forward"  : subset -> superset
                            "backward" : superset -> subset

        Returns:

        model_report (dict): models performance on each subset of features
        """
        try:
            # run model evaluation for all sub set of features
            model_report = dict()
            features_list = []
            model_names = []
            all_scores = []

            # Decide which models to run
            models_to_run = []

            if custom_model is not None:
                models_to_run.extend(custom_model)
            elif model_tags:
                for model_tag in model_tags:
                    model_name, model = self.get_model(model_tag)[model_tag]
                    models_to_run.append((model_name, model))
            else:
                raise ValueError("You must provide either `model_tags` or a `custom_model`.")

            # cross validation object
            cv = StratifiedKFold(n_splits=self.cv, random_state=self.seed, shuffle=self.shuffle)

            # feature selection run on models  
            for model_name, model in models_to_run:

                # features selection object
                selection_obj = RFECV(
                estimator=model,        # try on this model
                step=1,                     # num of features at a time
                cv=cv,                      # evaluation metric
                scoring=self.scoring,       # evaluation metric
                min_features_to_select=1,   # minimun feature select
                n_jobs=-1,                  # use all cores
                )
                
                # fitting object to select features
                selection_obj.fit(X, y)

                # selected features
                selected_features = selection_obj.get_feature_names_out(self.features)

                # collecting finally selected features
                features_list.append(selected_features)

                # collecting models name
                model_names.append(model_name)    

                # evaluating for score on selected features
                eval_scores = self.evaluate_model(model, selection_obj.get_support(indices=True))

                # keeping the track of model run and scores
                self.model_report[model_name] = eval_scores

                # collecting scores of each fold
                all_scores.append(eval_scores)

                # print scores
                if print_status:
                    # print model name
                    print('='*15,model_name,'='*15)

                    # mean scores
                    mean, std = self.get_custome_scores(eval_scores)

                    msg = f"{len(selected_features)} : Mean: {mean} Std: {std}\n"
                    print(msg)

            # collecting all scores
            model_report["model_name"] = model_names
            model_report["selected_features"] = features_list
            model_report["cv_scores"] = all_scores

            return model_report
        except Exception as e:
            raise e

In [ ]:
# models work with forward selection approach
model_tags = ["LR", "NB", "SVM", "KNN", "DT", "RF", "GB", "AB", "ETC", "XGB", "XGRFB","CB"]

### 2.2 Backward Feature Selection

In [ ]:
wrapper_bfs_obj = Wrapper_Methods(
    DATA,
    FEATURES,
    TARGET_COL,
    SCORING,
    CV,
    SEED,
    SHUFFLE
)

#### Logistic Regression

In [ ]:
model_report = wrapper_bfs_obj.directional_feature_selection(["LR"], True, "backward")

### Final Report

In [ ]:
bfs_models_report = wrapper_bfs_obj.get_scores(
    pd.concat(
    [   
        pd.DataFrame(LR_model_report),
        pd.DataFrame(NB_model_report),
        # pd.DataFrame(SVM_model_report),
        pd.DataFrame(KNN_model_report),
        pd.DataFrame(DT_model_report),
        pd.DataFrame(RF_model_report),
        pd.DataFrame(GB_model_report),
        pd.DataFrame(AB_model_report),
        pd.DataFrame(ETC_model_report),
        pd.DataFrame(XGB_model_report),
        pd.DataFrame(XGRFB_model_report),
        pd.DataFrame(CB_model_report)
    ],
    axis=0
).reset_index(
    drop=True
)
)

In [ ]:
bfs_models_report

In [ ]:
wrapper_bfs_obj.box_plot(bfs_models_report['model_name'].to_list(), bfs_models_report['cv_scores'].to_list())

### 2.3 Backward Feature Elimination

Possible Algorithms to use (only parametric algorithms):

- Logistic Regression
- Suport Vector Machine
- Decision Tree
- Random Forest
- Gradient Boost
- Ada Boost
- Extra Tree Classifier
- Xtream Gradient Boost
- Xtream Gradient Boost(Random Forest)
- Cat Boost

In [ ]:
wrapper_bfe_obj = Wrapper_Methods(
    DATA,
    FEATURES,
    TARGET_COL,
    SCORING,
    CV,
    SEED,
    SHUFFLE
)

#### 1. Logistic Regression

In [ ]:
LR_model_report = wrapper_bfe_obj.backward_feature_elimination(["LR"], True)
LR_model_report

#### 2. Support Vector Machine

In [ ]:
SVM_model_report = wrapper_bfe_obj.backward_feature_elimination(["SVM"], True)
SVM_model_report

#### 3. Decision Tree

In [ ]:
DT_model_report = wrapper_bfe_obj.backward_feature_elimination(["DT"], True)
DT_model_report

#### 4. Random Forest

In [ ]:
RF_model_report = wrapper_bfe_obj.backward_feature_elimination(["RF"], True)
RF_model_report

#### 5. Gradient Boost

In [ ]:
GB_model_report = wrapper_bfe_obj.backward_feature_elimination(["GB"], True)
GB_model_report

#### 6. Ada Boost

In [ ]:
AB_model_report = wrapper_bfe_obj.backward_feature_elimination(["AB"], True)
AB_model_report

#### 7. Extra Tree Classifier

In [ ]:
ETC_model_report = wrapper_bfe_obj.backward_feature_elimination(["ETC"], True)
ETC_model_report

#### 8. Xtream Gradient Boost

In [ ]:
XGB_model_report = wrapper_bfe_obj.backward_feature_elimination(["XGB"], True)
XGB_model_report

#### 9. Xtream Gradient Boost Random Forest

In [ ]:
XGRFB_model_report = wrapper_bfe_obj.backward_feature_elimination(["XGRFB"], True)
XGRFB_model_report

#### 10. Cat Boost

In [ ]:
CB_model_report = wrapper_bfe_obj.backward_feature_elimination(["CB"], True)
CB_model_report

### Final Report

In [ ]:
bfe_models_report = wrapper_bfe_obj.get_scores(
    pd.concat(
    [   
        pd.DataFrame(LR_model_report),
        pd.DataFrame(NB_model_report),
        # pd.DataFrame(SVM_model_report),
        pd.DataFrame(KNN_model_report),
        pd.DataFrame(DT_model_report),
        pd.DataFrame(RF_model_report),
        pd.DataFrame(GB_model_report),
        pd.DataFrame(AB_model_report),
        pd.DataFrame(ETC_model_report),
        pd.DataFrame(XGB_model_report),
        pd.DataFrame(XGRFB_model_report),
        pd.DataFrame(CB_model_report)
    ],
    axis=0
).reset_index(
    drop=True
)
)

In [ ]:
bfe_models_report

In [ ]:
wrapper_bfe_obj.box_plot(bfe_models_report['model_name'].to_list(), bfe_models_report['cv_scores'].to_list())

## 3. Embedded Methods

In [ ]:
import numpy as np

def plot_feature_importance(importances, feature_names, title="Feature Importances", top_n=None):
    """
    Plots a bar chart of feature importances.

    Args:
        importances (array-like): Array of importance scores or coefficients.
        feature_names (list): Corresponding feature names.
        title (str): Plot title.
        top_n (int or None): Number of top features to plot. If None, plot all.
    """
    importances = np.array(importances)
    feature_names = np.array(feature_names)

    # Filter top N if needed
    if top_n is not None:
        indices = np.argsort(importances)[-top_n:][::-1]
    else:
        indices = np.argsort(importances)[::-1]

    plt.figure(figsize=(10, 6))
    plt.barh(range(len(indices)), importances[indices], align='center')
    plt.yticks(range(len(indices)), feature_names[indices])
    plt.gca().invert_yaxis()
    plt.xlabel("Importance")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

### 3.1 Decision Tree

In [ ]:
# Train a Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_res, y_res)

In [ ]:
# Plot feature importances
plot_feature_importance(
    importances=dt.feature_importances_,
    feature_names=FEATURES,
    title="Decision Tree Feature Importances"
)

### 3.2 Random Forest

In [ ]:
# Train random forest
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_res, y_res)

In [ ]:
# Plot feature importances
plot_feature_importance(
    importances=rf.feature_importances_,
    feature_names=FEATURES,
    title="Random Forest Feature Importances"
)

**Conclusion:**

Selected Columns:

- `Age_Power`, `Avg_glucose_level_Power`, `Bmi_Imputed_median`, `Ever_married_encode`, `Gender_encode`, 
`Residence_type_encode`, `Work_type_encode`, `Work_type_children`, `Smonking_status_encode`

# Selected Features

In [ ]:
STATISTICALLY_FILTTERED = ['Age_Power', 'Avg_glucose_level_Power', 'Bmi_Imputed_median',
                            'Ever_married_encode', 'Work_type_encode', 'Work_type_children',
                            'Work_type_self_employed', 'Smoking_status_encode', 'Smoking_status_formerly_smoked', 
                            'Smoking_status_unknown']

In [ ]:
FORWARDS_FEATURE_SELECTION = ['Age_Power', 'Ever_married_encode', 'Work_type_encode',
         'Work_type_children', 'Work_type_govt_job',
         'Work_type_self_employed', 'Residence_type_encode',
         'Smoking_status_never_smoked']

In [ ]:
BACKWARDS_FEATURE_SELECTION = ['Age_Power', 'Avg_glucose_level_Power', 'Gender_encode',
         'Ever_married_encode', 'Work_type_never_worked',
         'Residence_type_encode', 'Smoking_status_formerly_smoked',
         'Smoking_status_never_smoked', 'Smoking_status_smokes']

In [ ]:
BACKWARD_FEATURE_ELIMINATION = ['Age_Power', 'Avg_glucose_level_Power', 'Gender_encode',
         'Ever_married_encode', 'Work_type_encode', 'Work_type_private',
         'Smoking_status_encode', 'Smoking_status_never_smoked']

In [ ]:
EBEDDED_FEATURES = ['Age_Power', 'Avg_glucose_level_Power', 'Bmi_Imputed_median', 'Ever_married_encode', 'Gender_encode', 
'Residence_type_encode', 'Work_type_encode', 'Work_type_children', 'Smonking_status_encode']